# SUDOKU FROM IMAGE EXTRACTOR

Alex Keyken 20/11/26

Detect the largest bounding quadrilateral in the image, add image processing,
transform into a square, extract the digit cells and draw into 28*28 pixel canvases.

Reviewed and optimized using Claude Code.

In [49]:
import os
from pathlib import Path

import cv2
import numpy as np

# Extract Sudoku Grid

In [51]:
def save_step(name: str, image: np.ndarray) -> None:
    """Save an image to the report directory."""
    REPORT_DIR = Path("reports")
    
    os.makedirs(REPORT_DIR, exist_ok=True)
    cv2.imwrite(REPORT_DIR / f"{name}.png", image)

def sort_corners(vertices: list[list[int]]) -> list[np.float32]:
    """Return an aranged list of box vertices clockwise starting from the top left corner."""
    vertices = vertices.reshape(4, 2)

    s = vertices.sum(axis=1)
    diff = np.diff(vertices, axis=1)

    top_left = vertices[np.argmin(s)]
    bottom_right = vertices[np.argmax(s)]
    top_right = vertices[np.argmin(diff)]
    bottom_left = vertices[np.argmax(diff)]

    return np.array([top_left, top_right, bottom_right, bottom_left], dtype=np.float32)

def find_sudoku_grid(image: np.ndarray, should_save_step: bool = False) -> list[np.float32]:
    """
    Find the vertice coords of the largest bounding box 
    in the image assumed to be the sudoku grid outline.
    """
    ADAPTIVE_THRESH_NEIGHBOURS = 11
    MIN_AREA = 1000
    
    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) # grayscale
    if should_save_step:
        save_step("01_grayscale",gray_img)
    
    blurred_img = cv2.GaussianBlur(gray_img, (7, 7), 0) # remove intense artefacts
    if should_save_step:
        save_step("02_gaussblur",blurred_img)

    # convert the image to black and white with adaptive thresholds (neighbour pixels contribute more)
    # invert colors so the background is black and contours white
    thresh_img = cv2.adaptiveThreshold(blurred_img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, ADAPTIVE_THRESH_NEIGHBOURS, 2)
    if should_save_step:
        save_step("03_adthresh",thresh_img)

    # keep only outermost contours and save to a list of corners
    contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if should_save_step:
        contour_img = thresh_img.copy()
        cv2.drawContours(contour_img, contours, -1, (0, 255, 0), 2)
        save_step("04_contours", contour_img)

    largest_quad,largest_area = None, 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        # ignore tiny contours
        if area < MIN_AREA:
            continue

        # approximate rough edged shape to a polygon with an error within 2% of the perimeter 
        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.10*peri, True)

        if len(approx) == 4 and area > largest_area:
            largest_area = area
            largest_quad = approx

    if largest_quad is None:
        raise RuntimeError("Could not find Sudoku grid")

    if should_save_step:
        corner_img = thresh_img.copy()
        cv2.drawContours(corner_img, [largest_quad], -1, (0, 0, 255), 5)
        save_step("05_detected_grid", corner_img)

    return sort_corners(largest_quad)

def warp_sudoku(image: np.ndarray, corners: list[np.float32],
                should_save_step: bool = False) -> image: np.ndarray:
    """Warp quadrilateral box from image into a square image with 90 deg corners."""
    PIX_PER_SIDE = 9*100 # cells * pixels

    corner_pixels = np.array([
            [0, 0],
            [PIX_PER_SIDE - 1, 0],
            [PIX_PER_SIDE - 1, PIX_PER_SIDE - 1],
            [0, PIX_PER_SIDE - 1],
        ], dtype=np.float32)

    transform = cv2.getPerspectiveTransform(corners, corner_pixels)
    warped = cv2.warpPerspective(image, transform, (PIX_PER_SIDE, PIX_PER_SIDE))

    if should_save_step:
        save_step("06_warped_grid", warped)

    return warped

In [46]:
image = cv2.imread("images/test_sudoku.jpg")
corners = find_sudoku_grid(image, should_save_step=True)
warped = warp_sudoku(image, corners, should_save_step=True)

# Extract Cells

In [41]:
def extract_cells(warped: np.ndarray) -> list[list[np.ndarray]]:
    """
    Extract digit cell images from larger sudoku grid image
    into a row*col array.
    """
    gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
    cell_size = gray.shape[0]//9

    cells = []
    for row in range(9):
        row_cells = []
        for col in range(9):
            y1 = row*cell_size
            y2 = (row + 1)*cell_size
            x1 = col*cell_size
            x2 = (col+1)*cell_size

            cell = gray[y1:y2, x1:x2]
            
            row_cells.append(cell)
            
        cells.append(row_cells)

    return cells

def clean_cell(cell: np.ndarray) -> np.ndarray:
    """
    Process rough cell extracted from sudoku grid, into a clean 28*28 pixel canvas
    containing the centred digit to be identified by the CNN.
    """
    h, w = cell.shape

    # crop cell to remove borders
    margin = int(min(h, w) * 0.15)
    cell = cell[margin:h-margin, margin:w-margin]
    
    # add blur to remove artefacts
    cell = cv2.GaussianBlur(cell, (3, 3), 0)
    
    """
    # convert grayscale to black and white
    _, cell = cv2.threshold(cell, 0, 255, cv2.THRESH_OTSU)

    # find digit pixels in non-empty cells
    inverted = cv2.bitwise_not(cell)
    coords = None
    if cv2.countNonZero(inverted) > 1000:
        coords = cv2.findNonZero(inverted)
        x, y, w, h = cv2.boundingRect(coords)
        cell = cell[y:y+h, x:x+w]        
    """
    
    # scale smallest side to min 20 pix avoiding major distortion
    scale = 20/max(h, w)
    new_w = int(w*scale)
    new_h = int(h*scale)
    cell = cv2.resize(cell, (new_w, new_h),
                      interpolation=cv2.INTER_AREA)

    # create canvas for 28*28 pixel output image
    canvas = np.zeros((28, 28), dtype=np.uint8)
    x_offset = (28 - new_w) // 2
    y_offset = (28 - new_h) // 2

    # add centred cropped digit
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = cell

    return canvas

In [47]:
# FOR DEBUGGING

grid_size = 28
cells = extract_cells(warped)
board = np.zeros((9*grid_size, 9*grid_size), dtype=np.uint8)
for row in range(9):
    for col in range(9):
        digit_img = clean_cell(cells[row][col])
        
        y1 = row*grid_size
        y2 = (row + 1)*grid_size
        x1 = col*grid_size
        x2 = (col + 1)*grid_size
        
        board[y1:y2, x1:x2] = digit_img

cv2.imwrite("reports/07_processed_digits.png", board)

True